In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json 

load_dotenv()

key = os.environ["OPENROUTER_API_KEY"]
if key is None: 
    raise ValueError("API key not found")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=key,
)
FREE_MODEL = "openrouter/free"
TOOL_MODEL = "openrouter/free"

response = client.chat.completions.create(
    model=FREE_MODEL,
    messages=[{"role": "user", "content": "What is 2 + 2? Answer in one word."}],
    temperature=0.0,
    max_tokens=50,
)
print("Response:", response.choices[0].message.content)
print(f"Tokens — in: {response.usage.prompt_tokens}, out: {response.usage.completion_tokens}")
print(f"Model used: {response.model}")

Response: Four
Tokens — in: 22, out: 2
Model used: liquid/lfm-2.5-1.2b-instruct-20260120:free


In [6]:
prompt = "write a sentece about AI"


for temp in [0.0, 1.0, 2.0]:
    results = []

    for _ in range (3):
        response = client.chat.completions.create(
            model=FREE_MODEL,
            messages=[{"role": "user", "content": prompt}], 
            temperature=temp,
            max_tokens=50
        )

        content = response.choices[0].message.content

        if content:
            results.append(content.strip())
        else:
            results.append("No content returned")

    print(f"Temp: {temp}")
    for i, text in enumerate(results):
        print(f"[{i+1}] {text[:120]}")
        

Temp: 0.0
[1] No content returned
[2] No content returned
[3] Artificial intelligence is rapidly evolving and transforming industries, from healthcare and finance to entertainment an
Temp: 1.0
[1] No content returned
[2] No content returned
[3] No content returned
Temp: 2.0
[1] Artificial intelligence is rapidly transforming industries and redefining what's possible, from automating tasks to driv
[2] No content returned
[3] Artificial intelligence is rapidly transforming industries and redefining what's possible, from automating tasks to driv


In [7]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city. Returns temperature and conditions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. 'Tokyo'"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression and return the result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression, e.g. '(5 + 3) * 2'"}
                },
                "required": ["expression"]
            }
        }
    }
]

In [9]:
r = client.chat.completions.create(
    model=TOOL_MODEL,
    messages=[{"role": "user", "content": "What's the weather in Tokyo?"}],
    tools=tools,
    temperature=0,
)

msg = r.choices[0].message
print(f"Finish reason: {r.choices[0].finish_reason}")
print(f"Content: {msg.content}")
print(f"Tool calls: {msg.tool_calls}")

if msg.tool_calls:
    tc = msg.tool_calls[0]
    print(f"\n→ Model wants to call: {tc.function.name}({tc.function.arguments})")
    print("  It GENERATED JSON requesting a function. Our code must execute it.")

Finish reason: tool_calls
Content: None
Tool calls: [ChatCompletionMessageFunctionToolCall(id='call_719a7147d07c4007b5617e08', function=Function(arguments='{"city": "Tokyo"}', name='get_weather'), type='function', index=0)]

→ Model wants to call: get_weather({"city": "Tokyo"})
  It GENERATED JSON requesting a function. Our code must execute it.


In [13]:
import json

def get_weather(city: str):
    fake = {
        "Tokyo": {"temp": "22°C", "condition": "partly cloudy"},
        "London": {"temp": "14°C", "condition": "rainy"},
        "Delhi": {"temp": "38°C", "condition": "sunny"},
    }
    return json.dumps(fake.get(city, {"temp": "unknown", "condition": "unknown"}))

def calculate(expression: str) -> int:
    allowed = set("0123456789+-*/.() ")
    try:
        if not all(c in allowed for c in expression):
            return json.dumps({"error": "Invalid expression"})
        return json.dumps({results: eval(expression)})
    except Exception as e:
        return json.dumps({"error": str(e)})

TOOL_FNS = {"get_weather": get_weather, "calculate": calculate}

if msg.tool_calls:
    tc = msg.tool_calls[0]
    fn = TOOL_FNS[tc.function.name]
    result = fn(**json.loads(tc.function.arguments))
    print(f"Tool result: {result}")

    messages = [
        {"role": "user", "content": "What's the weather in Tokyo?"},
        msg,
        {"role": "tool", "tool_call_id": tc.id, "content": result}
    ]

    r2 = client.chat.completions.create(
        model=TOOL_MODEL, messages=messages, tools=tools
    )
    print(f"\nFinal answer: {r2.choices[0].message.content}")

Tool result: {"temp": "22\u00b0C", "condition": "partly cloudy"}

Final answer: The weather in Tokyo is currently 22°C with partly cloudy conditions.


In [14]:
REACT_SYSTEM_PROMPT = """You are a helpful assistant that solves problems step by step using tools.

You have access to these tools:
{tool_descriptions}

## How to respond

When you need to use a tool, respond in EXACTLY this format:

THOUGHT: <your reasoning about what to do next>
ACTION: <tool_name>
ACTION_INPUT: <arguments as valid JSON>

When you have enough information for the final answer:

THOUGHT: <your final reasoning>
FINAL_ANSWER: <your complete answer to the user>

## Rules
- Always start with THOUGHT
- Use only ONE action per turn
- Wait for the OBSERVATION before continuing
- If a tool returns an error, reason about it and try a different approach
- Be concise in your thoughts
"""

In [ ]:
import requests

def search_wikipedia(query: str) -> str:
    query_clean = query.lower().replace(" ", "_")
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{query_clean}"

    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            return json.dumps({
                "title": data.get("title", ""),
                "summary": data.get("extract", "no summary found")[:800]
            })
        return json.dumps({ "error": "Page not found for {query}. Try a diff term" })
    except Exception as e:
        return json.dumps({ "error": str(e) })

def calculate_math(expression: str) -> int:
    try:
        allowed = set("0123456789.()/+-* eE")
        if not all(c in allowed for c in expression):
            return json.dumps({ "error": "invalid expression" })
        result = eval(expression)
        return json.dumps({ "expression": expression, "result": round(result, 6) })
    except Exception as e:
        return json.dumps({"error": str(e)})
TOOLS = {
    "search_wikipedia": {
        "fn": search_wikipedia,
        "desc": "search_wikipedia(query: str) — Search Wikipedia. Use simple topic names like 'France' or 'Albert Einstein'."
    },
    "calculate": {
        "fn": calculate_math,
        "desc": "calculate(expression: str) — Evaluate a math expression. Example: '(5 + 3) * 2.5'"
    }
}

print(f"Tools registered: {list(TOOLS.keys())}")

Tools registered: ['search_wikipedia', 'calculate']


In [ ]:
def run_agent(user_query: str, max_interations: int = 10, verbose: bool=True):
        tool_desc = "\n".join(f"- {t["desc"]}" for t in TOOLS.values())
        system = REACT_SYSTEM_PROMPT.format(tool_descriptions=tool_desc)

        messages = [
                {"role": "system", "content": system},
                {"role": "user", "content": user_query}
        ]

        if verbose: 
                print("=" * 60)
                print("user_query: ", user_query)
                print("=" * 60)

        for i in range(max_interations):
                if verbose:
                        print(f"\n ----Iteration{i+1}/{max_interations}----")          
        
                response = client.chat.completions.create(
                    model=FREE_MODEL,
                    messages=messages,
                    temperature=0,
                    max_tokens=800,
                )

                text = response.choices[0].message.content or ""
                messages.append({"role": "assistant", "content": text})       
        